## CELL INIT

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!apt-get install -y fonts-liberation -q

import os, json
import numpy as np
import matplotlib, matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from collections import defaultdict

BASE     = '/content/drive/MyDrive/adversarial_iot_paper'
SEC7_DIR = os.path.join(BASE, 'results', 'section7_v2')
FIG_DIR  = os.path.join(BASE, 'figures', 'section7_v2')
os.makedirs(FIG_DIR, exist_ok=True)

found = [f.name for f in fm.fontManager.ttflist if 'Liberation Sans' in f.name]
FONT  = 'Liberation Sans' if found else 'DejaVu Sans'
matplotlib.rcParams.update({
    'font.family'    : FONT,
    'font.size'      : 8,
    'axes.titlesize' : 8,
    'axes.labelsize' : 7.5,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 6.5,
    'axes.linewidth' : 0.6,
    'pdf.fonttype'   : 42,
    'font.weight'    : 'normal',
})
FIG_W = 6.93; DPI = 300

wb  = json.load(open(os.path.join(SEC7_DIR,'wb_results_v2.json')))
ca  = json.load(open(os.path.join(SEC7_DIR,'ca_results_v2.json')))
cd  = json.load(open(os.path.join(SEC7_DIR,'cd_results_v2.json')))
st  = json.load(open(os.path.join(SEC7_DIR,'stat_results_v2.json')))
def_= json.load(open(os.path.join(SEC7_DIR,'defense_results_v2.json')))

valid_wb = {k:v for k,v in wb.items() if 'error' not in v}
valid_ca = {k:v for k,v in ca.items() if 'error' not in v}
valid_cd = {k:v for k,v in cd.items() if 'error' not in v}

EPS_LIST = [0.01, 0.05, 0.1]
DATASETS = ['CIC-IDS 2017','UNSW-NB15','Gotham IoT 2025','CIC-YNU-IoTMal 2026']
# ВАЖЛИВО: 'XGBoost' — правильна назва у wb/ca/def результатах
MODELS   = ['RF','XGBoost','MLP','CNN']
C = {'RF':'#2878B5','XGBoost':'#C82423','MLP':'#FF8C00','CNN':'#6A5ACD'}
DS_SHORT = {
    'CIC-IDS 2017'       :'CIC-IDS\n2017',
    'UNSW-NB15'          :'UNSW-\nNB15',
    'Gotham IoT 2025'    :'Gotham\nIoT 2025',
    'CIC-YNU-IoTMal 2026':'CIC-YNU\nIoTMal\n2026',
}

def rsp(ax):
    for sp in ['top','right']: ax.spines[sp].set_visible(False)

def save_fig(fig, name):
    p = os.path.join(FIG_DIR, name)
    if os.path.exists(p): os.remove(p)
    fig.savefig(p, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
    kb = os.path.getsize(p)/1024
    print(f'✓ Saved: {name} ({kb:.0f} KB)')

# Перевірка назв моделей
model_names = set(v['model'] for v in valid_wb.values())
print(f'✓ INIT | Font: {FONT}')
print(f'  Model names in results: {model_names}')
print(f'  wb={len(valid_wb)} ca={len(valid_ca)} cd={len(valid_cd)}')

## CELL STATUS

In [ ]:
print('=== Section 7 Figures STATUS ===')
expected = [f'fig{i}' for i in range(6,13)]
for i in range(6,13):
    files = [f for f in os.listdir(FIG_DIR)
             if f.startswith(f'fig{i}_') and f.endswith('.png')]
    if files:
        for f in sorted(files):
            kb = os.path.getsize(os.path.join(FIG_DIR,f))/1024
            print(f'  ✓  {f} ({kb:.0f} KB)')
    else:
        print(f'  —  Fig {i}: not generated')

## CELL F6 — Figure 6: ASR vs epsilon (MLP and CNN)
Легенда: прозора, нижній правий кут — не перекриває криві.

In [ ]:
OUT6 = os.path.join(FIG_DIR,'fig6_asr_vs_eps.png')

PANELS = [('MLP','CIC-IDS 2017','(a)'),
          ('CNN','CIC-IDS 2017','(b)'),
          ('MLP','UNSW-NB15',   '(c)'),
          ('CNN','UNSW-NB15',   '(d)')]

ATK_STYLE = {
    'FGSM': {'color':'#2878B5','ls':'-', 'marker':'o','ms':3.5},
    'PGD' : {'color':'#C82423','ls':'--','marker':'s','ms':3.5},
    'CW'  : {'color':'#FF8C00','ls':':' ,'marker':'^','ms':3.5},
}

fig,axes = plt.subplots(2,2,figsize=(FIG_W, FIG_W*0.78))
fig.subplots_adjust(hspace=0.50, wspace=0.38)

for idx,(mn,ds,panel) in enumerate(PANELS):
    ax = axes[idx//2][idx%2]
    for atk in ['FGSM','PGD','CW']:
        asr_vals = []
        for eps in EPS_LIST:
            v = valid_wb.get(f'{ds}|{mn}|{atk}|{eps}',{})
            asr_vals.append(v.get('asr', np.nan))
        s = ATK_STYLE[atk]
        ax.plot(EPS_LIST, asr_vals, color=s['color'],
                ls=s['ls'], marker=s['marker'], ms=s['ms'],
                lw=0.9, label=atk)

    ax.set_xlabel('Perturbation budget ε', fontsize=7, labelpad=2)
    ax.set_ylabel('ASR', fontsize=7, labelpad=2)
    ax.set_title(f'{panel} {mn} — {ds}', fontsize=8, pad=4, loc='left')
    ax.set_xticks(EPS_LIST)
    ax.set_xlim(0.005, 0.115)
    ax.set_ylim(-0.02, 1.08)
    # Легенда прозора (framealpha=0.3) у нижньому правому
    ax.legend(fontsize=6, loc='lower right', handlelength=1.5,
              framealpha=0.3, edgecolor='#aaaaaa', borderpad=0.4)
    ax.tick_params(labelsize=6, length=2.5, width=0.5)
    rsp(ax)

save_fig(fig,'fig6_asr_vs_eps.png')

## CELL F7 — Figure 7: White-box ASR heatmap

In [ ]:
OUT7 = os.path.join(FIG_DIR,'fig7_wb_asr_heatmap.png')

asr_dict = defaultdict(list)
for v in valid_wb.values():
    asr_dict[(v['model'],v['ds'],v['attack'])].append(v['asr'])
mean_asr = {k:np.mean(vs) for k,vs in asr_dict.items()}

fig,axes = plt.subplots(1,3,figsize=(FIG_W, 2.6))
fig.subplots_adjust(wspace=0.38)

for idx,atk in enumerate(['FGSM','PGD','CW']):
    ax = axes[idx]
    mat = np.full((len(MODELS),len(DATASETS)), np.nan)
    for i,mn in enumerate(MODELS):
        for j,ds in enumerate(DATASETS):
            val = mean_asr.get((mn,ds,atk))
            if val is not None: mat[i,j] = val

    # Маскуємо NaN (для RF/XGB де немає CW)
    mat_plot = np.where(np.isnan(mat), 0, mat)
    im = ax.imshow(mat_plot, cmap='Reds', vmin=0, vmax=1, aspect='auto')

    ax.set_xticks(range(len(DATASETS)))
    ax.set_xticklabels([DS_SHORT[d] for d in DATASETS],
                       fontsize=5, ha='center', linespacing=1.1)
    ax.set_yticks(range(len(MODELS)))
    # XGBoost → коротко XGB на осі
    ax.set_yticklabels(['RF','XGB','MLP','CNN'], fontsize=6)
    ax.set_title(f'({chr(97+idx)}) {atk}', fontsize=8, pad=4, loc='left')

    for i in range(len(MODELS)):
        for j in range(len(DATASETS)):
            key = (MODELS[i], DATASETS[j], atk)
            if key in mean_asr:
                v = mat[i,j]
                clr = 'white' if v > 0.55 else 'black'
                ax.text(j,i,f'{v:.2f}',ha='center',va='center',
                        fontsize=5,color=clr)
            else:
                # N/A для RF/XGB без CW
                ax.text(j,i,'N/A',ha='center',va='center',
                        fontsize=4.5,color='#aaaaaa')

    plt.colorbar(im,ax=ax,fraction=0.046,pad=0.04).ax.tick_params(labelsize=5)

save_fig(fig,'fig7_wb_asr_heatmap.png')

## CELL F8 — Figure 8: White-box vs Black-box ASR

In [ ]:
OUT8 = os.path.join(FIG_DIR,'fig8_wb_vs_bb.png')

WB_MODELS = ['MLP','CNN']
BB_MODELS  = ['RF','XGBoost']

fig,axes = plt.subplots(1,2,figsize=(FIG_W, 2.6))
fig.subplots_adjust(wspace=0.42)
x = np.arange(len(DATASETS)); w = 0.35

for idx,atk in enumerate(['FGSM','PGD']):
    ax = axes[idx]
    wb_means,bb_means = [],[]
    for ds in DATASETS:
        wb_v = [valid_wb.get(f'{ds}|{mn}|{atk}|{eps}',{}).get('asr',np.nan)
                for mn in WB_MODELS for eps in EPS_LIST]
        bb_v = [valid_wb.get(f'{ds}|{mn}|{atk}|{eps}',{}).get('asr',np.nan)
                for mn in BB_MODELS for eps in EPS_LIST]
        wb_means.append(np.nanmean(wb_v))
        bb_means.append(np.nanmean(bb_v))

    ax.bar(x-w/2, wb_means, w, color='#2878B5', alpha=0.85,
           label='White-box (MLP/CNN)', edgecolor='white', lw=0.3)
    ax.bar(x+w/2, bb_means, w, color='#C82423', alpha=0.85,
           label='Black-box surrogate (RF/XGB)', edgecolor='white', lw=0.3)

    for i,(wv,bv) in enumerate(zip(wb_means,bb_means)):
        ax.text(i-w/2, wv+0.012, f'{wv:.2f}',
                ha='center', va='bottom', fontsize=5.5)
        ax.text(i+w/2, bv+0.012, f'{bv:.2f}',
                ha='center', va='bottom', fontsize=5.5)

    ax.set_xticks(x)
    ax.set_xticklabels([DS_SHORT[d] for d in DATASETS],
                       fontsize=5.5, ha='center', linespacing=1.1)
    ax.set_ylabel('Mean ASR (averaged over ε)', fontsize=7, labelpad=2)
    ax.set_ylim(0, 0.90)
    ax.set_title(f'({chr(97+idx)}) {atk}', fontsize=8, pad=4, loc='left')
    ax.legend(fontsize=5.5, loc='upper right', framealpha=0.85)
    ax.tick_params(labelsize=6, length=2.5, width=0.5)
    rsp(ax)

save_fig(fig,'fig8_wb_vs_bb.png')

## CELL F9 — Figure 9: Cross-architecture TR heatmap

In [ ]:
OUT9 = os.path.join(FIG_DIR,'fig9_cross_arch_tr.png')

SRC = ['MLP','CNN']
TGT = ['RF','XGBoost','MLP','CNN']
TGT_LABELS = ['→RF','→XGB','→MLP','→CNN']

fig,axes = plt.subplots(1,2,figsize=(FIG_W, 2.6))
fig.subplots_adjust(wspace=0.52)

for idx,atk in enumerate(['FGSM','PGD']):
    ax = axes[idx]
    mat = np.full((len(SRC),len(TGT)), np.nan)

    for i,src in enumerate(SRC):
        for j,tgt in enumerate(TGT):
            if src == tgt: continue
            trs = []
            for ds in DATASETS:
                for eps in EPS_LIST:
                    v = valid_ca.get(f'{ds}|{src}→{tgt}|{atk}|{eps}',{})
                    if v.get('tr') is not None:
                        trs.append(v['tr'])
            if trs: mat[i,j] = np.mean(trs)

    # Для діагональних (src==tgt) — сірий колір
    diag_mask = np.zeros_like(mat,dtype=bool)
    for i,src in enumerate(SRC):
        for j,tgt in enumerate(TGT):
            if src==tgt: diag_mask[i,j]=True

    mat_plot = np.where(diag_mask, 1.0, np.where(np.isnan(mat),1.0,mat))
    im = ax.imshow(mat_plot, cmap='RdBu_r', vmin=0.0, vmax=2.0, aspect='auto')

    # Сіре закрашення діагональних комірок
    for i in range(len(SRC)):
        for j in range(len(TGT)):
            if SRC[i]==TGT[j]:
                ax.add_patch(plt.Rectangle((j-0.5,i-0.5),1,1,
                    fc='#e0e0e0',ec='white',lw=0.5))

    ax.set_xticks(range(len(TGT)))
    ax.set_xticklabels(TGT_LABELS, fontsize=6.5)
    ax.set_yticks(range(len(SRC)))
    ax.set_yticklabels(SRC, fontsize=6.5)
    ax.set_xlabel('Target model', fontsize=7, labelpad=2)
    ax.set_ylabel('Source model', fontsize=7, labelpad=2)
    ax.set_title(f'({chr(97+idx)}) {atk} — TR (mean over ε, datasets)',
                 fontsize=8, pad=4, loc='left')

    for i in range(len(SRC)):
        for j in range(len(TGT)):
            if SRC[i]==TGT[j]:
                ax.text(j,i,'—',ha='center',va='center',
                        fontsize=7,color='#888888')
            elif not np.isnan(mat[i,j]):
                v = mat[i,j]
                clr = 'white' if abs(v-1.0)>0.45 else 'black'
                ax.text(j,i,f'{v:.2f}',ha='center',va='center',
                        fontsize=6.5,color=clr)

    cb = plt.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
    cb.ax.tick_params(labelsize=5.5)
    cb.set_label('TR', fontsize=6)
    cb.ax.axhline(y=0.5,color='black',lw=0.8,ls='--')

save_fig(fig,'fig9_cross_arch_tr.png')

## CELL F10 — Figure 10: Cross-dataset adversarial transfer

In [ ]:
OUT10 = os.path.join(FIG_DIR,'fig10_cross_dataset.png')

AL_MODELS  = ['RF','XGB','MLP']
DIRECTIONS = [
    ('CIC-IDS 2017','UNSW-NB15',  '(a) CIC-IDS 2017 → UNSW-NB15'),
    ('UNSW-NB15','CIC-IDS 2017',  '(b) UNSW-NB15 → CIC-IDS 2017'),
]

fig,axes = plt.subplots(1,2,figsize=(FIG_W, 2.8))
fig.subplots_adjust(wspace=0.42)

for idx,(src,tgt,title) in enumerate(DIRECTIONS):
    ax = axes[idx]
    x  = np.arange(len(AL_MODELS)); w = 0.35

    fgsm_asr,pgd_asr = [],[]
    for mn in AL_MODELS:
        f_v = [valid_cd.get(f'{src}→{tgt}|{mn}|FGSM|{eps}',{}).get('asr_tgt',np.nan)
               for eps in EPS_LIST]
        p_v = [valid_cd.get(f'{src}→{tgt}|{mn}|PGD|{eps}',{}).get('asr_tgt',np.nan)
               for eps in EPS_LIST]
        fgsm_asr.append(np.nanmean(f_v))
        pgd_asr.append(np.nanmean(p_v))

    ax.bar(x-w/2, fgsm_asr, w, color='#2878B5', alpha=0.85,
           label='FGSM', edgecolor='white', lw=0.3)
    ax.bar(x+w/2, pgd_asr,  w, color='#C82423', alpha=0.85,
           label='PGD',  edgecolor='white', lw=0.3)

    for i,(fv,pv) in enumerate(zip(fgsm_asr,pgd_asr)):
        if not np.isnan(fv):
            ax.text(i-w/2, fv+0.008, f'{fv:.3f}',
                    ha='center', va='bottom', fontsize=5)
        if not np.isnan(pv):
            ax.text(i+w/2, pv+0.008, f'{pv:.3f}',
                    ha='center', va='bottom', fontsize=5)

    ax.set_xticks(x)
    ax.set_xticklabels(AL_MODELS, fontsize=7)
    ax.set_ylabel('ASR on target dataset (mean over ε)', fontsize=7, labelpad=2)
    ax.set_ylim(0, 0.75)
    ax.set_title(title, fontsize=8, pad=4, loc='left')
    ax.legend(fontsize=6, loc='upper left', framealpha=0.85)
    ax.tick_params(labelsize=6, length=2.5, width=0.5)
    ax.axhline(y=0.01, color='#888888', lw=0.7, ls=':', label='_nolegend_')
    rsp(ax)

save_fig(fig,'fig10_cross_dataset.png')

## CELL F11 — Figure 11: Robustness Score
Підписи всередині стовпчиків. XGBoost включено.

In [ ]:
OUT11 = os.path.join(FIG_DIR,'fig11_robustness_score.png')

rs_data     = st.get('robustness_scores',{})
rs_by_model = defaultdict(list)
rs_by_ds    = defaultdict(list)
for v in rs_data.values():
    rs_by_model[v['model']].append(v['rs'])
    rs_by_ds[v['ds']].append(v['rs'])

print('RS keys:', {k:len(v) for k,v in rs_by_model.items()})

fig,axes = plt.subplots(1,2,figsize=(FIG_W, 3.4))
fig.subplots_adjust(wspace=0.48)

# Panel (a): RS by model
ax = axes[0]
means    = [np.mean(rs_by_model[m]) if rs_by_model.get(m) else 0 for m in MODELS]
stds     = [np.std(rs_by_model[m])  if rs_by_model.get(m) else 0 for m in MODELS]
colors_m = [C[m] for m in MODELS]

bars = ax.bar(range(len(MODELS)), means, color=colors_m,
              alpha=0.88, edgecolor='white', lw=0.3)
ax.errorbar(range(len(MODELS)), means, yerr=stds,
            fmt='none', color='#333333', lw=0.9,
            capsize=3, capthick=0.8)
for bar,v in zip(bars,means):
    if v > 0.08:
        ax.text(bar.get_x()+bar.get_width()/2, v/2,
                f'{v:.3f}', ha='center', va='center',
                fontsize=6, color='white')
ax.set_xticks(range(len(MODELS)))
# Скорочені підписи — XGBoost → XGB
ax.set_xticklabels(['RF','XGB','MLP','CNN'], fontsize=7)
ax.set_ylabel('Robustness Score (RS)', fontsize=7, labelpad=2)
ax.set_ylim(0, 1.12)
ax.set_title('(a) RS by model architecture', fontsize=8, pad=4, loc='left')
ax.tick_params(labelsize=6, length=2.5, width=0.5)
rsp(ax)

# Panel (b): RS by dataset
ax = axes[1]
ds_colors = ['#4CAF50','#2196F3','#FF9800','#9C27B0']
means_ds  = [np.mean(rs_by_ds[d]) if rs_by_ds.get(d) else 0 for d in DATASETS]
stds_ds   = [np.std(rs_by_ds[d])  if rs_by_ds.get(d) else 0 for d in DATASETS]

bars = ax.bar(range(len(DATASETS)), means_ds, color=ds_colors,
              alpha=0.88, edgecolor='white', lw=0.3)
ax.errorbar(range(len(DATASETS)), means_ds, yerr=stds_ds,
            fmt='none', color='#333333', lw=0.9,
            capsize=3, capthick=0.8)
for bar,v in zip(bars,means_ds):
    if v > 0.08:
        ax.text(bar.get_x()+bar.get_width()/2, v/2,
                f'{v:.3f}', ha='center', va='center',
                fontsize=6, color='white')
ax.set_xticks(range(len(DATASETS)))
ax.set_xticklabels([DS_SHORT[d] for d in DATASETS],
                   fontsize=6, ha='center', linespacing=1.2)
ax.set_ylabel('Robustness Score (RS)', fontsize=7, labelpad=2)
ax.set_ylim(0, 1.12)
ax.set_title('(b) RS by dataset', fontsize=8, pad=4, loc='left')
ax.tick_params(labelsize=6, length=2.5, width=0.5)
rsp(ax)

save_fig(fig,'fig11_robustness_score.png')

## CELL F12 — Figure 12: Feature Squeezing defense

In [ ]:
OUT12 = os.path.join(FIG_DIR,'fig12_defense_fs.png')

DS_DEF = ['CIC-IDS 2017','UNSW-NB15']
# Правильні назви як у defense_results
COMBOS = [(mn,atk) for mn in ['CNN','MLP','RF','XGBoost']
                   for atk in ['FGSM','PGD']]

fig,axes = plt.subplots(2,1,figsize=(FIG_W, 5.0))
fig.subplots_adjust(hspace=0.54)

for idx,ds_name in enumerate(DS_DEF):
    ax = axes[idx]
    x  = np.arange(len(COMBOS)); w = 0.25

    clean_vals,before_vals,after_vals,labels = [],[],[],[]
    for mn,atk in COMBOS:
        v = def_.get(f'{ds_name}|{mn}|{atk}',{})
        clean_vals.append(v.get('acc_clean', np.nan))
        before_vals.append(v.get('asr_before', np.nan))
        after_vals.append(v.get('asr_fs', np.nan))
        labels.append(f'{mn}\n{atk}')

    ax.bar(x-w, clean_vals,  w, color='#4CAF50', alpha=0.85,
           label='Clean accuracy', edgecolor='white', lw=0.3)
    ax.bar(x,   before_vals, w, color='#C82423', alpha=0.85,
           label='ASR before FS', edgecolor='white', lw=0.3)
    ax.bar(x+w, after_vals,  w, color='#FF9800', alpha=0.85,
           label='ASR after FS', edgecolor='white', lw=0.3)

    # Підписи всередині after_FS стовпчиків
    for i,av in enumerate(after_vals):
        if not np.isnan(av) and av > 0.05:
            ax.text(x[i]+w, av/2, f'{av:.2f}',
                    ha='center', va='center', fontsize=5,
                    color='white')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=5.5, ha='center', linespacing=1.2)
    ax.set_ylabel('Rate', fontsize=7, labelpad=2)
    ax.set_ylim(0, 1.32)
    ax.set_title(f'({chr(97+idx)}) {ds_name}', fontsize=8, pad=4, loc='left')
    if idx==0:
        ax.legend(fontsize=6, loc='upper right', ncol=3,
                  framealpha=0.85, edgecolor='#cccccc')
    ax.tick_params(labelsize=6, length=2.5, width=0.5)
    rsp(ax)

save_fig(fig,'fig12_defense_fs.png')

## CELL VERIFY — Фінальна перевірка всіх рисунків

In [ ]:
print('=== Section 7 Figures FINAL STATUS ===')
expected = [
    'fig6_asr_vs_eps.png',
    'fig7_wb_asr_heatmap.png',
    'fig8_wb_vs_bb.png',
    'fig9_cross_arch_tr.png',
    'fig10_cross_dataset.png',
    'fig11_robustness_score.png',
    'fig12_defense_fs.png',
]
all_ok = True
for fname in expected:
    p = os.path.join(FIG_DIR, fname)
    if os.path.exists(p):
        kb = os.path.getsize(p)/1024
        print(f'  ✓  {fname:<45} {kb:>6.0f} KB')
    else:
        print(f'  ✗  {fname} — MISSING'); all_ok = False
print()
print('✓ All 7 figures ready.' if all_ok else '⚠ Run missing CELL above.')